# Naming the Topics of the 1914 Press

BERTopic described the topics of *L'Humanité* in 1914 by their keywords, without naming them. Here, a local LLM names them: the course's serving with Ollama and structured output, on a real corpus.

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs: a smaller LLM (Qwen3-1.7B instead of Qwen3-4B), slower on a CPU, then names the 5 largest topics only.

## Data

Ollama installs and downloads the LLM in the background.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q "bertopic<1"

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# On a GPU, Qwen3-4B. On a CPU, Qwen3-1.7B, twice as fast: a "hybrid" model, which
# thinks before answering, except with reasoning_effort="none" (in the calls).
MODEL = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b-q4_K_M"
# On a GPU, 8 requests at a time; on a CPU, one: each request processed in parallel
# reserves its own context in memory (Ollama's FAQ).
parallel = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": parallel, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
download = subprocess.Popen(
  f"sleep 5 && ollama pull {MODEL}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

# The installation is silent: a printed version shows that it succeeded.
print(
  subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
)

## The Topics

This cell is given: it redoes the topics of the lab "Topics of the 1914 Press", with its settings. They are the same 20,000 passages of *L'Humanité*, with their precomputed *embeddings* (Qwen3-Embedding-0.6B, truncated to 512 dimensions), reduced to 5 dimensions by UMAP, with topics of at least 50 passages. The outlier passages are then reassigned to their closest topic. BERTopic then warns that this reassignment must be the last step: here, it is. It takes a few minutes, while the LLM downloads.

In [ ]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité only (Le Figaro has only 24 issues), and a sample of 20,000 passages so
# that the computation takes a few minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
vectors = embeddings[humanite.index]

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)

# The outlier passages join their closest topic.
new_topics = topic_model.reduce_outliers(
  docs, topics, strategy="embeddings", embeddings=vectors
)
topic_model.update_topics(
  docs,
  topics=new_topics,
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
)
info = topic_model.get_topic_info().sort_values("Count", ascending=False)
info[["Topic", "Count", "Representation"]].head(10)

## Name the Topics with an LLM

Keywords describe a topic, but don't name it. For each of the `N_TOPICS` largest topics (20; without a GPU, 5), give the LLM its 10 keywords and its 3 representative passages (their first 300 characters), and ask it for a short name, 3 to 6 words, in English. The client of the frame "Serving a Model Yourself" is ready: call `client.chat.completions.parse` for an output constrained to the `TopicName` format (frame "Constraining the Output"; spell the format out in the prompt: the model doesn't see the schema). Send the requests in parallel, set the names with `topic_model.set_topic_labels(names)` (a topic → name dictionary), and display `get_topic_info()`. Are the names right? Which one would you fix by hand?

**Useful documentation**

- [`chat.completions.parse`](https://github.com/openai/openai-python/blob/main/helpers.md#auto-parsing-response-content-with-pydantic-models)
- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [Ollama's OpenAI API](https://docs.ollama.com/api/openai-compatibility)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
- [`BERTopic.get_topic`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.set_topic_labels`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels)

In [ ]:
download.wait()  # the model is downloaded

from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# A first call loads the model (onto the GPU if there is one), where it stays: before any parallel work.
client.chat.completions.create(
  model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1
)
print(MODEL, "ready")


class TopicName(BaseModel):
  name: str


# The topics to name: 20 on a GPU; without one, 5 (one to two minutes).
N_TOPICS = 20 if GPU else 5

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `topic_model.get_topic(topic)[:10]`: the first 10 (keyword, weight) pairs; `topic_model.get_representative_docs(topic)[:3]`: 3 passages, of which we keep `doc[:300]`
- the prompt spells the format out: `Answer with a JSON object: {"name": "..."}`; the schema constrains the output, but the model doesn't see it, and it sometimes wanders off into whitespace before the JSON
- `client.chat.completions.parse(model=MODEL, messages=[{"role": "user", "content": prompt}], response_format=TopicName, temperature=0, reasoning_effort="none", max_tokens=100)`; the name is in `response.choices[0].message.parsed.name`
- `reasoning_effort="none"`: the model answers without a thinking phase (without a GPU, Qwen3-1.7B would have one, which multiplies the response time)
- the `N_TOPICS` largest topics: after the reassignment, there is no topic −1 left, and `get_topic_info()` follows the topics' numbers: sort it with `sort_values("Count", ascending=False)`, then `info.Topic[:N_TOPICS].tolist()`
- `ThreadPoolExecutor(8)`: 8 calls in parallel; `pool.map(name_topic, largest)` returns the names in order; `dict(zip(largest, names))`
- `topic_model.set_topic_labels(names)`: the names appear in the `CustomName` column of `get_topic_info()`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def name_topic(topic: int) -> str:
  keywords = ", ".join(word for word, _ in topic_model.get_topic(topic)[:10])
  examples = "\n\n".join(
    doc[:300] for doc in topic_model.get_representative_docs(topic)[:...]
  )
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[
      {
        "role": "user",
        "content": (
          "Here is a topic of the 1914 French press, found automatically.\n"
          f"Its keywords: {keywords}\n\nThree representative passages:\n{examples}\n\n"
          "Give it a short, precise name, 3 to 6 words, in English.\n"
          'Answer with a JSON object: {"name": "..."}'
        ),
      }
    ],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
    max_tokens=100,
  )
  return response.choices[0].message.parsed.name


info = topic_model.get_topic_info().sort_values("Count", ascending=False)
largest = info.Topic[:...].tolist()
with ThreadPoolExecutor(8) as pool:
  names = dict(zip(largest, pool.map(..., largest)))
topic_model.set_topic_labels(...)
info = topic_model.get_topic_info().sort_values("Count", ascending=False)
info[["Topic", "Count", "CustomName", "Representation"]].head(N_TOPICS)
```

### Solution

In [ ]:
def name_topic(topic: int) -> str:
  keywords = ", ".join(word for word, _ in topic_model.get_topic(topic)[:10])
  examples = "\n\n".join(
    doc[:300] for doc in topic_model.get_representative_docs(topic)[:3]
  )
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[
      {
        "role": "user",
        "content": (
          "Here is a topic of the 1914 French press, found automatically.\n"
          f"Its keywords: {keywords}\n\nThree representative passages:\n{examples}\n\n"
          "Give it a short, precise name, 3 to 6 words, in English.\n"
          # The schema constrains the output, but the model doesn't see it: without this
          # reminder, it sometimes wanders off into whitespace before the JSON.
          'Answer with a JSON object: {"name": "..."}'
        ),
      }
    ],
    response_format=TopicName,
    temperature=0,
    reasoning_effort="none",
    max_tokens=100,
  )
  return response.choices[0].message.parsed.name


info = topic_model.get_topic_info().sort_values("Count", ascending=False)
largest = info.Topic[:N_TOPICS].tolist()
with ThreadPoolExecutor(8) as pool:
  names = dict(zip(largest, pool.map(name_topic, largest)))
topic_model.set_topic_labels(names)
info = topic_model.get_topic_info().sort_values("Count", ascending=False)
info[["Topic", "Count", "CustomName", "Representation"]].head(N_TOPICS)

# BERTopic can also do it itself: representation_model=bertopic.representation.OpenAI(client,
# model=MODEL), with any OpenAI-compatible client. Always read the names over: an LLM names a
# clear-cut topic well, a catch-all topic badly.